In [ ]:
import os
import re
from datetime import datetime
from itertools import chain
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import spacy
from spacy.tokens.doc import Doc
from spacy.tokens.span import Span
from spacy.tokens.token import Token
import coreferee
from coreferee.data_model import ChainHolder
from helpers import get_request
from bs4 import BeautifulSoup
from bs4.element import Tag

In [ ]:
nlp = spacy.load("en_core_web_lg")
nlp.add_pipe("coreferee")

In [ ]:
def get_text_from_file(path: str):
    text = ""
    
    try:
        with open(path, "r", encoding="utf8") as file:
            text = file.read()
    except Exception as e:
        print("Error reading file", path)
        print(e)
    finally:
        return text

In [ ]:
articles_folder = "./articles"

In [ ]:
article_files = os.listdir(articles_folder)
article_files = [file for file in article_files if file.endswith(".txt")]

In [ ]:
texts = [get_text_from_file(f"{articles_folder}/{file}") for file in article_files]

df = pd.DataFrame(data={
    "file": article_files,
    "text": texts
})
df

In [ ]:
def already_split_gender_paragraphs(article_file: str):
    gender_paragraphs_folder = f"{articles_folder}/{article_file.replace('.txt', '')}"
    return os.path.exists(gender_paragraphs_folder)

In [ ]:
# Since this part of the pipeline is very resource intensive, 
# we only want to run it on articles that haven't already been split on gender.
# However, you can set this variable to True if you still want to run on everything.
run_on_all_files = False

if run_on_all_files:
    df_to_separate = df
else:
    df_to_separate = df[~df["file"].apply(already_split_gender_paragraphs)].copy()

df_to_separate

In [ ]:
# This takes a long time, so have patience
df_to_separate["doc"] = list(nlp.pipe(df_to_separate["text"]))

In [ ]:
def get_people(doc: Doc):
    return [ent for ent in doc.ents if ent.label_ == "PERSON"]

In [ ]:
df_to_separate["people"] = df_to_separate["doc"].apply(get_people)

In [ ]:
all_people: list[Span] = list(chain.from_iterable(df_to_separate["people"]))
all_names = set(p.text for p in all_people)
full_names = set(name for name in all_names 
                 if 
                 len(name.split()) >= 2 
                 and len(name.split()) <= 3
                 and re.search("^[a-zA-Z\u00C0-\u017F\s’]+$", name) is not None)

full_names

In [ ]:
def is_woman_tag(tag: Tag):
    text = tag.get_text().strip().lower()
    return "woman" in text or "women" in text

def find_gender_on_ufc(name: str):
    name_query = "-".join(name.split()).replace('’', ' ')
    url = f"https://www.ufc.com/athlete/{name_query}"

    try:
        response = get_request(url)
        soup = BeautifulSoup(response.text, "html.parser")

        tags: list[Tag] = soup.find_all("p", "hero-profile__tag")
        if not tags:
            return None

        is_woman = any([is_woman_tag(tag) for tag in tags])
        if is_woman:
            return "woman"
        else:
            return "man"
    except Exception as e:
        print("Error getting gender of", name)
        print(e)

In [ ]:
# Set this to False if you want to fetch the genders from the UFC website
use_saved_gender_map = False
genders_map_file = "./genders_map.csv"

if use_saved_gender_map and os.path.exists(genders_map_file):
    name_gender_pairs_df = pd.read_csv(genders_map_file)
    genders_map: dict[str, str] = dict(zip(name_gender_pairs_df["name"], name_gender_pairs_df["gender"]))
else:
    def get_name_gender_pair_on_ufc(name: str):
        return (name, find_gender_on_ufc(name))

    with ThreadPoolExecutor() as executor:
        name_gender_pairs = [
            (n, g)
            for n, g in executor.map(get_name_gender_pair_on_ufc, full_names)
            if g is not None
        ]

    genders_map = dict(name_gender_pairs)

    for name, gender in name_gender_pairs:
        for name_component in name.split():
            genders_map[name_component] = gender

    pd.DataFrame(data={
        "name": genders_map.keys(),
        "gender": genders_map.values(),
    })\
        .set_index("name")\
        .to_csv(f"./genders_map_{datetime.now().strftime('%d-%m-%y_%H-%M-%S')}.csv")

genders_map

In [ ]:
def get_coreference_genders(row: pd.Series):
    doc: Doc = row["doc"]

    chains: ChainHolder = doc._.coref_chains
    
    def loop():
        for chain in chains:
            main_item_mention = chain[chain.most_specific_mention_index]
            main_item = doc[main_item_mention.root_index]
            
            if main_item.text in genders_map:
                for mention in chain:
                    token = doc[mention.root_index]
                    yield (token, genders_map[main_item.text])
    
    return list(loop())

In [ ]:
df_to_separate["coreference_genders"] = df_to_separate.apply(get_coreference_genders, axis="columns")

In [ ]:
def get_people_with_gender(row: pd.Series, gender: str):
    doc: Doc = row["doc"]
    people: list[Span] = row["people"]
    coreference_genders: list[tuple[Token, str]] = row["coreference_genders"]

    people_with_gender = [p for p in people if genders_map.get(p.text) == gender]
    coreferences_with_gender = [p for p, g in coreference_genders if g == gender]
    coreferences_with_gender = [Span(doc, p.i, p.i + 1) for p in coreferences_with_gender]
    
    return set(people_with_gender + coreferences_with_gender)

In [ ]:
df_to_separate["men"] = df_to_separate.apply(get_people_with_gender, args=("man",), axis="columns")
df_to_separate["women"] = df_to_separate.apply(get_people_with_gender, args=("woman",), axis="columns")

In [ ]:
def get_paragraph_boundaries(doc: Doc):
    line_breaks = [token for token in doc if token.text == "\n"]
    line_break_indexes = [lb.i for lb in line_breaks]
    paragraph_boundaries = list(zip([0] + line_break_indexes, line_break_indexes + [len(doc)]))
    return paragraph_boundaries

In [ ]:
df_to_separate["paragraph_boundaries"] = df_to_separate["doc"].apply(get_paragraph_boundaries)

In [ ]:
def count_genders_in_paragraphs(row: pd.Series, gender: str):
    people_with_gender: set[Span] = row[gender]
    paragraph_boundaries: list[tuple[int, int]] = row["paragraph_boundaries"]

    def count_gender_in_paragraph(boundary: tuple[int, int]):
        start, end = boundary
        count = 0

        for person in people_with_gender:
            if person.end >= end:
                continue
            elif person.start < start:
                continue
            
            count += 1

        return count

    return [count_gender_in_paragraph(boundary) for boundary in paragraph_boundaries]

In [ ]:
df_to_separate["men_per_paragraph"] = df_to_separate.apply(count_genders_in_paragraphs, args=("men",), axis="columns")
df_to_separate["women_per_paragraph"] = df_to_separate.apply(count_genders_in_paragraphs, args=("women",), axis="columns")

In [ ]:
def assign_gender_to_paragraphs(row: pd.Series):
    men_per_paragraph: list[int] = row["men_per_paragraph"]
    women_per_paragraph: list[int] = row["women_per_paragraph"]

    def choose_gender(m: int, f: int):
        if m == 0 and f == 0:
            return None
        elif m > f:
            return "man"
        elif f > m:
            return "woman"
        else: 
            return "equal"

    return [choose_gender(m, f) for m, f in zip(men_per_paragraph, women_per_paragraph)]

In [ ]:
df_to_separate["paragraph_genders"] = df_to_separate.apply(assign_gender_to_paragraphs, axis="columns")

In [ ]:
def get_paragraphs_with_gender(row: pd.Series, gender: str):
    doc: Doc = row["doc"]
    paragraph_boundaries: list[tuple[int, int]] = row["paragraph_boundaries"]
    paragraph_genders: list[str | None] = row["paragraph_genders"]

    paragraphs_indexes_with_gender = [i for i, g in enumerate(paragraph_genders) if g == gender]
    paragraphs_boundaries_with_gender = [paragraph_boundaries[i] for i in paragraphs_indexes_with_gender]
    paragraphs_with_gender = [doc[start:end] for start, end in paragraphs_boundaries_with_gender]

    return paragraphs_with_gender

In [ ]:
df_to_separate["man_paragraphs"] = df_to_separate.apply(get_paragraphs_with_gender, args=("man",), axis="columns")
df_to_separate["woman_paragraphs"] = df_to_separate.apply(get_paragraphs_with_gender, args=("woman",), axis="columns")
df_to_separate["equal_paragraphs"] = df_to_separate.apply(get_paragraphs_with_gender, args=("equal",), axis="columns")
df_to_separate["genderless_paragraphs"] = df_to_separate.apply(get_paragraphs_with_gender, args=(None,), axis="columns")
df_to_separate

In [ ]:
def save_separate_paragraphs(row: pd.Series):
    original_file: str = row["file"]
    new_folder = original_file.replace(".txt", "")

    os.makedirs(f"./articles/{new_folder}", exist_ok=True)

    man_paragraphs: list[Span] = row["man_paragraphs"]
    man_text = "\n".join([p.text for p in man_paragraphs])

    with open(f"./articles/{new_folder}/man.txt", "w", encoding="utf8", newline="\n") as file:
        file.write(man_text)

    woman_paragraphs: list[Span] = row["woman_paragraphs"]
    woman_text = "\n".join([p.text for p in woman_paragraphs])

    with open(f"./articles/{new_folder}/woman.txt", "w", encoding="utf8", newline="\n") as file:
        file.write(woman_text)

    equal_paragraphs: list[Span] = row["equal_paragraphs"]
    equal_text = "\n".join([p.text for p in equal_paragraphs])

    with open(f"./articles/{new_folder}/equal.txt", "w", encoding="utf8", newline="\n") as file:
        file.write(equal_text)

    genderless_paragraphs: list[Span] = row["genderless_paragraphs"]
    genderless_text = "\n".join([p.text for p in genderless_paragraphs])

    with open(f"./articles/{new_folder}/genderless.txt", "w", encoding="utf8", newline="\n") as file:
        file.write(genderless_text)

In [ ]:
df_to_separate.apply(save_separate_paragraphs, axis="columns")